# Analysis — Health Inequality in Sepsis Care

Investigates whether treatment patterns documented in discharge notes differ across demographic groups (race, insurance) in a sepsis cohort. NER-extracted entities from MIMIC-IV discharge summaries are aggregated at the patient level and analyzed using descriptive statistics and regression.

**No patient data is written to disk or displayed in outputs.**

In [50]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

In [51]:
entities = pd.read_parquet("../data/processed/entities.parquet")
merged = pd.read_parquet("../data/processed/merged_sepsis_notes.parquet")

print(entities.shape)
print(entities["label"].value_counts())
print(entities["entity"].value_counts())
print(entities["section"].value_counts())

(1113051, 12)
label
treatment    618091
problem      447879
test          47081
Name: count, dtype: int64
entity
pain                            21082
aspirin                         18453
acetaminophen                   15619
docusate sodium                 12882
constipation                    10638
                                ...  
fs                                  1
d10 gtt                             1
the patient ' s blood sugars        1
pentoxifylline cr                   1
intussception                       1
Name: count, Length: 124707, dtype: int64
section
brief hospital course       364310
medications on admission    296195
discharge medications       289417
discharge diagnosis         163129
Name: count, dtype: int64


## Patient-Level Aggregation

In [52]:
# long form for section-level analysis
treatment_counts = (
    entities[entities["label"] == "treatment"]
    .groupby(["hadm_id", "section"])["entity"]
    .count()
    .unstack(fill_value=0)
    .reset_index()
)
treatment_counts.columns.name = None

# add total
treatment_counts["total_treatments"] = treatment_counts.drop(columns="hadm_id").sum(axis=1)

# problem counts
problem_counts = (
    entities[entities["label"] == "problem"]
    .groupby("hadm_id")["entity"]
    .count()
    .reset_index()
    .rename(columns={"entity": "problem_count"})
)

# now patient_df is one row per hadm_id
patient_df = treatment_counts.merge(problem_counts, on="hadm_id", how="left")
patient_df = patient_df.merge(
    merged[["hadm_id", "subject_id", "insurance", "race", "gender",
            "anchor_age", "los", "hospital_expire_flag"]],
    on="hadm_id",
    how="left"
)



## Demographics
MIMIC race values are granular and inconsistent, so I group into broad categories for analysis.

In [53]:
race_map = {
    # White
    "WHITE": "White",
    "WHITE - OTHER EUROPEAN": "White",
    "WHITE - RUSSIAN": "White",
    "WHITE - EASTERN EUROPEAN": "White",
    "WHITE - BRAZILIAN": "White",
    "PORTUGUESE": "White",

    # Black
    "BLACK/AFRICAN AMERICAN": "Black",
    "BLACK/CAPE VERDEAN": "Black",
    "BLACK/CARIBBEAN ISLAND": "Black",
    "BLACK/AFRICAN": "Black",

    # Hispanic
    "HISPANIC/LATINO - PUERTO RICAN": "Hispanic",
    "HISPANIC OR LATINO": "Hispanic",
    "HISPANIC/LATINO - DOMINICAN": "Hispanic",
    "HISPANIC/LATINO - GUATEMALAN": "Hispanic",
    "HISPANIC/LATINO - SALVADORAN": "Hispanic",
    "HISPANIC/LATINO - COLUMBIAN": "Hispanic",
    "HISPANIC/LATINO - CUBAN": "Hispanic",
    "HISPANIC/LATINO - HONDURAN": "Hispanic",
    "HISPANIC/LATINO - CENTRAL AMERICAN": "Hispanic",
    "HISPANIC/LATINO - MEXICAN": "Hispanic",
    "SOUTH AMERICAN": "Hispanic",

    # Asian
    "ASIAN": "Asian",
    "ASIAN - CHINESE": "Asian",
    "ASIAN - SOUTH EAST ASIAN": "Asian",
    "ASIAN - ASIAN INDIAN": "Asian",
    "ASIAN - KOREAN": "Asian",

    # Other
    "AMERICAN INDIAN/ALASKA NATIVE": "Other",
    "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER": "Other",
    "MULTIPLE RACE/ETHNICITY": "Other",
    "OTHER": "Other",

    # Unknown — set to None to drop later
    "UNKNOWN": None,
    "UNABLE TO OBTAIN": None,
    "PATIENT DECLINED TO ANSWER": None,
}

insurance_map = {
    "Medicare": "Medicare",
    "Medicaid": "Medicaid",
    "Private": "Private",
    "Other": "Other",
    "No charge": None,  # 1 patient, drop
}

# apply mappings
patient_df["race_group"] = patient_df["race"].map(race_map)
patient_df["insurance_group"] = patient_df["insurance"].map(insurance_map)


# sanity check
print(patient_df["hadm_id"].nunique(), len(patient_df))  # should be equal
print(patient_df["race_group"].value_counts(dropna=False))
print(patient_df["insurance_group"].value_counts(dropna=False))

30683 32373
race_group
White       22121
NaN          3606
Black        3332
Hispanic     1192
Other        1179
Asian         943
Name: count, dtype: int64
insurance_group
Medicare    19219
Private      7508
Medicaid     4446
Other         783
NaN           417
Name: count, dtype: int64


## Descriptive Statistics

In [54]:
# treatment counts by insurance
print("=== Treatments by Insurance ===")
print(
    patient_df.groupby("insurance_group")["total_treatments"]
    .agg(["mean", "median", "std", "count"])
    .round(2)
)

# treatment counts by race
print("\n=== Treatments by Race ===")
print(
    patient_df.groupby("race_group")["total_treatments"]
    .agg(["mean", "median", "std", "count"])
    .round(2)
)

# treatment ratio — intensity of care
patient_df["treatment_ratio"] = patient_df["total_treatments"] / patient_df["problem_count"]

print("=== Treatment Ratio by Insurance ===")
print(
    patient_df.groupby("insurance_group")["treatment_ratio"]
    .agg(["mean", "median", "std"])
    .round(3)
)

# section-level breakdown — where are the differences coming from
section_cols = ["brief hospital course", "medications on admission", 
                         "discharge medications", "discharge diagnosis"]

print("\n=== Treatments by Section and Insurance ===")
print(
    patient_df.groupby("insurance_group")[section_cols]
    .mean()
    .round(2)
)

# problem count by group — is severity actually comparable?
print("\n=== Problem Count by Insurance (severity check) ===")
print(
    patient_df.groupby("insurance_group")["problem_count"]
    .agg(["mean", "median", "std"])
    .round(2)
)

# outcomes
print("\n=== Mortality Rate by Insurance ===")
print(
    patient_df.groupby("insurance_group")["hospital_expire_flag"]
    .agg(["mean", "count"])
    .round(3)
)

print("\n=== Length of Stay by Insurance ===")
print(
    patient_df.groupby("insurance_group")["los"]
    .agg(["mean", "median", "std"])
    .round(2)
)

# age distribution — important confounder
print("\n=== Age by Insurance ===")
print(
    patient_df.groupby("insurance_group")["anchor_age"]
    .agg(["mean", "median", "std"])
    .round(2)
)

# gender breakdown
print("\n=== Gender by Insurance ===")
print(
    patient_df.groupby(["insurance_group", "gender"])
    .size()
    .unstack(fill_value=0)
)

=== Treatments by Insurance ===
                  mean  median    std  count
insurance_group                             
Medicaid         20.02    19.0  11.42   4446
Medicare         22.55    22.0  11.46  19219
Other            18.52    16.0  13.46    783
Private          19.92    18.0  12.09   7508

=== Treatments by Race ===
             mean  median    std  count
race_group                             
Asian       19.42    18.0   9.87    943
Black       22.01    21.0  12.05   3332
Hispanic    20.85    20.0  11.01   1192
Other       21.26    21.0  11.36   1179
White       21.84    21.0  11.68  22121
=== Treatment Ratio by Insurance ===
                  mean  median    std
insurance_group                      
Medicaid         1.701   1.238  1.817
Medicare         2.106   1.417  2.538
Other            1.760   1.214  1.946
Private          2.270   1.429  2.884

=== Treatments by Section and Insurance ===
                 brief hospital course  medications on admission  \
insurance_gr

## Visualizations

## Statistical Testing

## Regression Analysis
